# In what ways do Coding Agents contribute tests? 
#### How frequently do Coding Agents contribute tests? 
#### What types (e.g., unit, integration, end-to-end) are most common?

We weill explore the types of tests coding agents contribute. 

In [ ]:
import pandas as pd
import ollama
pr_task_type = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_task_type.parquet")

In [51]:
SYSTEM_PROMPT = """
You are a highly accurate classifier for software pull requests and commit messages.

Input fields:
- "title": a short PR or commit title
- "reason": an optional explanation of what the PR does

Your task:
Classify the PR into EXACTLY ONE category from this list:

["unit_tests", "integration_tests", "end_to_end_tests", "smoke_tests", "performance_tests", "regression_tests", "none"]

TITLE PRIORITY:
- The TITLE contains the strongest clues.
- If the title clearly references tests, ALWAYS choose a test-related category rather than "none".
- Words that indicate testing activity: "test", "tests", "unit", "coverage", "e2e", "end-to-end", "integration", "cypress", "jest", "pytest", "migrated test", "add tests", "improve tests", "test suite", "test cases".
- If the title starts with "Test", "test:", "tests:", or includes “test”, classify it as testing work unless the reason explicitly contradicts this.

REASON LOGIC:
- Use the reason if it adds helpful detail about the test *type*.
- If the reason is generic, missing, or unclear, rely primarily on the title.
- If the reason mentions fixing a previously known bug, choose "regression_tests".
- If the reason only describes CI, tooling, pre-commit hooks, formatting, or workflows unrelated to tests, choose "none".

CATEGORY DEFINITIONS:

unit_tests
- Tests for individual functions, classes, components, or isolated modules.
- Includes adding missing tests, increasing test coverage, migrating unit test frameworks, or expanding test suites for a single component.

integration_tests
- Tests involving interaction between multiple modules, services, or systems.
- Clues: API + database, service-to-service integration, multi-step internal pipelines.

end_to_end_tests
- Full-system user flows from start to finish.
- Clues: e2e, end-to-end, full workflow, user journey, UI-to-backend, Cypress flows.

smoke_tests
- Minimal/basic tests verifying that the system starts up or that major features work at all.

performance_tests
- Tests for speed, throughput, latency, load, benchmarking, scalability, stress.

regression_tests
- Tests added specifically to prevent a previously known bug, crash, or regression.
- DO NOT choose regression_tests just because the title mentions "error" or "fail".
- Only choose regression_tests if the PR explicitly references:
  - a past bug
  - a crash that occurred before
  - a fix from a previous commit or PR
  - a scenario meant to guard against recurrence

none
- Use ONLY when the PR does not add, modify, or meaningfully change tests.
- Examples:
  - CI config changes
  - pre-commit or workflow updates
  - linting/formatting
  - typo fixes
  - tooling-only updates
- DO NOT choose "none" when the title clearly indicates tests are added or modified.

TIE-BREAKING RULES:
- If title indicates test changes, prefer a test-related category over "none".
- If unsure between unit and integration, choose unit unless multiple systems interact.
- If unsure between integration and end_to_end, choose end_to_end only when the workflow clearly spans a full user-facing or end-to-end path.
- If unsure between unit and regression, choose regression only when previously-known bug or regression is explicitly mentioned.

OUTPUT FORMAT:
- Respond with ONLY the label.
- No punctuation, no quotes, no explanation, no extra words.
"""

In [50]:
# Explore the dataset
print(pr_task_type.shape)
print(pr_task_type.columns)
print(pr_task_type["type"].unique())

(33596, 6)
Index(['agent', 'id', 'title', 'reason', 'type', 'confidence'], dtype='object')
['fix' 'feat' 'chore' 'docs' 'test' 'refactor' 'build' 'ci' 'perf'
 'revert' 'style' 'other']


In [56]:
# Prompt was generated with the help of ChatGPT
PROMPT = SYSTEM_PROMPT

# Function which classifies PR in test into categories
def classify_pr(title: str, reason: str, model="qwen2.5-coder"):
    prompt = f"Title: {title}\nReason: {reason}"
    resp = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return resp["message"]["content"].strip()

In [91]:
import time

# Number of Samples and Sample Sizes
NUM_SAMPLES = 15
SAMPLE_SIZE = 5

samples = []
pr_test =  pr_task_type[pr_task_type["type"] == "test"]

# Generate samples ad benchmark time taken

start_benchmark = time.time()

for i in range(NUM_SAMPLES):
    start_sample = time.time()

    nth_sample = []
    for index, row in pr_test.sample(n = SAMPLE_SIZE).iterrows():
        title = row.title
        reason = row.reason
        nth_sample.append((title, reason, classify_pr(title, reason)))
    samples.append(nth_sample)

    end_sample = time.time()

    print(f"Time for Sample {i + 1}: {round(end_sample - start_sample, 2)} seconds")

end_benchmark = time.time()
print(f"Time: {round(end_benchmark - start_benchmark, 2)} seconds")

Time for Sample 1: 9.12 seconds
Time for Sample 2: 2.24 seconds
Time for Sample 3: 2.48 seconds
Time for Sample 4: 2.24 seconds
Time for Sample 5: 3.12 seconds
Time for Sample 6: 2.26 seconds
Time for Sample 7: 2.35 seconds
Time for Sample 8: 2.52 seconds
Time for Sample 9: 2.16 seconds
Time for Sample 10: 2.12 seconds
Time for Sample 11: 2.48 seconds
Time for Sample 12: 2.02 seconds
Time for Sample 13: 1.93 seconds
Time for Sample 14: 1.9 seconds
Time for Sample 15: 2.19 seconds
Time: 41.13 seconds


In [92]:
for index, s in enumerate(samples):
    print(f"Sample {index + 1}")
    for n in s:
        print("\t", f"{n[2]:<10} {n[0]}")
        print("\t", f"{' ':<10} {n[1]}")
        print()

Sample 1
	 unit_tests Add Kotlin load/save json golden test
	            The PR adds a new golden test for the Kotlin backend, which is related to testing functionality rather than fixing a bug or adding a feature.

	 unit_tests Update Pester tests and add coverage for tverrec functions
	            The changes involve fixing test helper variables and adding new test suites, which are related to testing improvements and corrections rather than features or bug fixes in the main codebase.

	 unit_tests Add Swift leetcode example tests
	            The PR adds tests for Swift leetcode examples and includes a helper to compile and run programs, which is focused on testing functionality rather than adding features or fixing bugs.

	 unit_tests Add test coverage for --aspire-version template option
	            The PR adds new test cases to cover the '--aspire-version' template option, which is explicitly about adding tests to ensure functionality works correctly. This fits the 'test' catego

We have the following sores for correctly classified tests of each Sample (size 5)

5, 5, 4, 3, 5, 5, 5, 4, 4, 5, 5, 5, 3, 5, 5

We have ~ 90 % accuracy here.

In the next step of the project, we will run the classifier on all ~2300 PRs classified as `test` in the dataset.

In [98]:
results = pd.DataFrame()
results["type"] = None
start_sample = time.time()
for i in range(len(pr_test)):
    title = row.title
    reason = row.reason
    results["type"] = results["type"].append(pd.Series([classify_pr(title, reason)]), ignore_index = True)
    end_sample = time.time()
    print(f"Time for Sample {i + 1}: {round(end_sample - start_sample, 2)} seconds")

end_benchmark = time.time()
print(f"Time: {round(end_benchmark - start_benchmark, 2)} seconds")

/var/folders/k_/gz_r13m967sd2js3rg57413c0000gn/T/ipykernel_17754/2544870981.py:7: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results["type"] = results["type"].append(pd.Series([classify_pr(title, reason)]), ignore_index = True)
/var/folders/k_/gz_r13m967sd2js3rg57413c0000gn/T/ipykernel_17754/2544870981.py:7: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results["type"] = results["type"].append(pd.Series([classify_pr(title, reason)]), ignore_index = True)


Time for Sample 1: 0.24 seconds
Time for Sample 2: 0.43 seconds
Time for Sample 3: 0.63 seconds
Time for Sample 4: 0.82 seconds
Time for Sample 5: 1.02 seconds
Time for Sample 6: 1.21 seconds
Time for Sample 7: 1.41 seconds
Time for Sample 8: 1.6 seconds
Time for Sample 9: 1.79 seconds
Time for Sample 10: 2.0 seconds
Time for Sample 11: 2.2 seconds
Time for Sample 12: 2.4 seconds
Time for Sample 13: 2.59 seconds
Time for Sample 14: 2.79 seconds
Time for Sample 15: 2.98 seconds
Time for Sample 16: 3.17 seconds
Time for Sample 17: 3.36 seconds
Time for Sample 18: 3.57 seconds
Time for Sample 19: 3.79 seconds
Time for Sample 20: 4.0 seconds
Time for Sample 21: 4.2 seconds
Time for Sample 22: 4.4 seconds
Time for Sample 23: 4.59 seconds
Time for Sample 24: 4.79 seconds
Time for Sample 25: 4.98 seconds
Time for Sample 26: 5.18 seconds
Time for Sample 27: 5.37 seconds
Time for Sample 28: 5.56 seconds
Time for Sample 29: 5.76 seconds
Time for Sample 30: 5.95 seconds
Time for Sample 31: 6.14 s

In [101]:
print(results)

         type
0  unit_tests
